# A Model-Agnostic Framework for Natural Language Explanations (NLE) of Predictive Models
**Master's Thesis Experimentation Notebook**

This notebook demonstrates a modular, object-oriented framework designed to translate the outputs of black-box predictive models into human-readable **Natural Language Explanations (NLEs)**.

*Note: To maintain a clean experimental environment, the heavy architectural logic has been abstracted into a dedicated Python package (`nle-framework-pappppx`), published on PyPI. This allows the notebook to focus strictly on data processing, prompt engineering, and experimental execution.*

### Research Hypothesis
The core objective is to evaluate whether injecting **Logic-based (Araucana-XAI)** alongside traditional **Feature Importance (SHAP, LIME)** improves the causal reasoning of Large Language Models (LLMs) and reduces hallucinations.

By comparing a *Baseline* (Feature Weights only) against an *Enhanced Hybrid* (Weights + Decision Tree Logic), we aim to demonstrate that providing explicit structural boundaries allows the LLM to generate more accurate, actionable, and causally grounded medical reports.

---
## 1. Environment Setup & Dependencies
In this section, we prepare the workspace to ensure exact reproducibility. Instead of relying on manual cloning or local requirement files, we install the framework directly from PyPI. This streamlined approach automatically resolves and installs the precise versions of the LLM inference engine (`unsloth` for efficient 4-bit quantized Llama-3 execution) and the Explainable AI (XAI) backends (`shap`, `lime`, and `araucanaxai`) required for the NLE pipeline.

In [ ]:
%%capture
# BLOCK 1: Environment Setup & Dependency Installation
# This block installs the NLE Framework directly from PyPI.
# All dependencies (SHAP, LIME, Araucana, Unsloth, etc.) are handled automatically.

import warnings
warnings.filterwarnings('ignore')

# 1. Install the framework from PyPI
!pip install nle-framework-pappppx

# NOTE: Unsloth is highly dependent on specific CUDA and PyTorch builds.
# If Google Colab updates its environment and the default installation fails,
# uncomment the following lines to force a fresh repository build:
# !pip install --upgrade --no-cache-dir "git+https://github.com/unslothai/unsloth-zoo.git"
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

## 2. Clinical Dataset & Black-Box Modeling
To simulate a high-stakes decision environment, we utilize the **Breast Cancer Wisconsin Diagnostic Dataset**.

We train a **Random Forest Classifier** as our surrogate "black-box" model. While Random Forests offer some intrinsic interpretability, their ensemble nature makes local, patient-specific decision paths complex to interpret without external post-hoc explanation tools.

**Pipeline Steps:**
1. Load and split the clinical dataset (80% Train / 20% Test).
2. Train the Random Forest algorithm.
3. Validate baseline accuracy to ensure the model is producing meaningful predictions before we attempt to explain them.

In [ ]:
# BLOCK 2: Data Pipeline & Black-Box Model Training
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Configuration
RANDOM_SEED = 42

def setup_medical_data():
    """
    Loads the Breast Cancer Wisconsin dataset.
    Returns: X_train, X_test, y_train, y_test, class_names, feature_names
    """
    data = load_breast_cancer()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['target'] = data.target

    X = df.drop('target', axis=1)
    y = df['target']

    # Split data (80% Train, 20% Test)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED
    )

    return X_train, X_test, y_train, y_test, data.target_names, data.feature_names

# 1. Load Data
print("Loading Medical Dataset...")
X_train, X_test, y_train, y_test, class_names, feature_names = setup_medical_data()

# 2. Train Black-Box Model (Random Forest)
# We use RF as it provides good performance but requires explanation tools (not intrinsically interpretable).
print("Training Random Forest Classifier...")
model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

# 3. Basic Evaluation
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f">>> Model Accuracy: {acc:.4f}")
print(f">>> Class Mapping: 0 = {class_names[0]}, 1 = {class_names[1]}")

## 3. Prompt Engineering: The Adapter Layer (Standardized)
To ensure experimental rigor, we enforce a **Standardized Prompt Structure** across all conditions. All adapters share the exact same definitions for `SYSTEM ROLE`, `OBJECTIVE`, `CONSTRAINTS`, `TONE & STYLE`, and the critical `HANDLING "BENIGN" CASES` protocol.

The only variable manipulated is the **Narrative Structure**, which dictates *how* the explanation is constructed:

### Set 1: Hybrid Adapters (SHAP + LIME + ARAUCANA)
* **Logic:** "Rule-Based Causality".
* **Narrative Instruction:** The LLM is instructed to identify the "Decisive Rule" or "Checklist" (Araucana) as the causal trigger, using SHAP/LIME only to confirm the intensity of the signal.

### Set 2: Baseline Adapters (SHAP + LIME Only)
* **Logic:** "Weighted Influence".
* **Narrative Instruction:** The LLM is instructed to explain the outcome based on "Cumulative Weights", "Feature Dominance", or "Tipping the Scale". References to logical paths or decision trees are strictly forbidden.

This structural parity ensures that any difference in the quality or hallucination rate of the generated text can be attributed to the **informational content** (Symbols vs. Weights) rather than prompt inconsistencies.

In [ ]:
# BLOCK 3: Adapter Configuration (Prompt Engineering)

# ==============================================================================
# 1. COMMON COMPONENTS (Standardized for ALL Prompts)
# ==============================================================================

# A. BENIGN CASE HANDLING (Clinical Safety)
BENIGN_INSTRUCTION = """
HANDLING "BENIGN" CASES
If the prediction is BENIGN: Do not use words like "Risk Factors". Refer to features as "Safety Signals" or "indicators of normalcy".
Example: "The benign classification is supported by low values in cellular perimeter and the absence of concave irregularities."
"""

# B. UNIVERSAL USER TEMPLATE (Single Input Structure)
# Since the input data format is identical for all personas, we use a single template.
# The System Prompt defines HOW to interpret this data.
UNIVERSAL_USER_TEMPLATE = """
PREDICTION: {prediction}

XAI CONTEXT (EVIDENCE):
{context_data}

PATIENT DATA (INPUTS):
{input_features}

Generate the response following the System Role instructions:
"""

# ==============================================================================
# 2. SET 1: HYBRID PROMPTS (SHAP + LIME + ARAUCANA)
# Logic: "Rule-Based Causality" (The Tree/Checklist defined the outcome).
# ==============================================================================

detailed_hybrid_system = f"""SYSTEM ROLE
Act as a Senior Computational Oncologist. Your task is to analyze the output of a predictive AI model and write a technical decision-support report.

OBJECTIVE
Synthesize a causal explanation combining the "Logic Path" (Araucana) with "Feature Importance" (SHAP/LIME).

INPUT DATA
You will receive Prediction, Combined XAI Context, and Patient Data.

{BENIGN_INSTRUCTION}

CONSTRAINTS
- NO LISTS OR BULLET POINTS. Use continuous professional prose.
- Do not dump raw rules; narrate the logic.

NARRATIVE STRUCTURE
1. Diagnostic Rationale: Start with the DECISIVE RULE from Araucana. "The model finalized this diagnosis primarily BECAUSE the patient's [Feature A] value of [Value] fell into the critical range..."
2. Structural Validation: Merge this with SHAP/LIME intensity. "This rule is statistically reinforced by the high positive impact (+0.X) of [Feature A]..."
3. The Logic Trace: Briefly narrate the checklist steps (e.g., "The algorithm initially stratified based on Size, then refined via Texture...").
4. Conclusion: Brief wrap-up.

TONE & STYLE
Terminology: Precise medical terms (morphology, cytology).
Structure: Dense, information-rich paragraphs.
Perspective: Third-person, objective.
"""

# ------------------------------------------------------------------------------

executive_hybrid_system = f"""SYSTEM ROLE
Clinical AI Assistant for a Medical Review Board.

OBJECTIVE
Provide a rapid, high-impact "Executive Summary" (<200 words) of the diagnostic logic.

INPUT DATA
You will receive Prediction, Combined XAI Context, and Patient Data.

{BENIGN_INSTRUCTION}

CONSTRAINTS
- NO LISTS. Single fluid paragraph.
- Focus on the "Decisive Moment" (The specific rule that sealed the verdict).

NARRATIVE STRUCTURE
1. The Verdict: State prediction and confidence.
2. The Causal Trigger: "This outcome was determined mainly BECAUSE [Feature X] exceeded the threshold of [Value]..." (Derived from Araucana).
3. The Consensus: "This structural logic is strongly supported by feature importance metrics, confirming [Feature X] as the dominant driver."

TONE & STYLE
Terminology: Executive, concise, decisive.
Structure: One summary paragraph.
Perspective: Third-person.
"""

# ------------------------------------------------------------------------------

patient_hybrid_system = f"""SYSTEM ROLE
Compassionate Medical Liaison explaining results to a patient.

OBJECTIVE
Explain the result using a "Checklist" analogy (derived from the Decision Tree).

INPUT DATA
You will receive Prediction, Combined XAI Context, and Patient Data.

{BENIGN_INSTRUCTION}

CONSTRAINTS
- NO JARGON (No "SHAP", "Nodes", "Vectors").
- Empathy First.

NARRATIVE STRUCTURE
1. Result Context: Clear and gentle.
2. The "Why" (The Checklist Analogy): "The system followed a specific checklist. First, it checked the size of the cells. Since that was [High/Low], it looked at the texture..."
3. The Cause: "Because your results showed pattern X followed by pattern Y, the system flagged this."
4. Next Steps: Encourage discussing these specific patterns with the doctor.

TONE & STYLE
Terminology: Simple visual descriptions (Shape, Size, Edges).
Structure: Warm, reassuring letter format.
Perspective: First-person ("We", "The system").
"""

# ==============================================================================
# 3. SET 2: BASELINE PROMPTS (SHAP + LIME ONLY)
# Logic: "Weighted Influence" (The Scale/Heatmap defined the outcome).
# ==============================================================================

detailed_baseline_system = f"""SYSTEM ROLE
Act as a Senior Computational Oncologist. Your task is to analyze the output of a predictive AI model based on FEATURE IMPORTANCE metrics.

OBJECTIVE
Synthesize an explanation based solely on "Statistical Weight" and "Feature Importance".

INPUT DATA
You will receive Prediction, XAI Weights, and Patient Data.

{BENIGN_INSTRUCTION}

CONSTRAINTS
- NO LISTS OR BULLET POINTS.
- DO NOT mention decision trees, rules, or logic paths (You lack this info).

NARRATIVE STRUCTURE
1. Diagnostic Rationale: Start with the DOMINANT BIOMARKERS. "The prediction is driven by a strong cumulative signal from [Feature A] and [Feature B]..."
2. Weighted Evidence: "Feature A exerts the highest positive impact (+0.X SHAP), suggesting it is the primary risk factor."
3. Local Confirmation: "Locally (LIME), the specific value confirms this trend."

TONE & STYLE
Terminology: Precise medical terms.
Structure: Dense, information-rich paragraphs.
Perspective: Third-person, objective.
"""

# ------------------------------------------------------------------------------

executive_baseline_system = f"""SYSTEM ROLE
Clinical AI Assistant for a Medical Review Board.

OBJECTIVE
Provide a rapid, high-impact "Executive Summary" (<200 words) based on Feature Dominance.

INPUT DATA
You will receive Prediction, XAI Weights, and Patient Data.

{BENIGN_INSTRUCTION}

CONSTRAINTS
- NO LISTS. Single fluid paragraph.
- NO Structural Logic (No "If-Then" rules).

NARRATIVE STRUCTURE
1. The Verdict: State prediction.
2. The Primary Driver: "The model prioritized this diagnosis CAUSED BY the high accumulated weight of [Feature X] and [Feature Y]."
3. The Evidence: "These features showed the strongest statistical correlation with the class."

TONE & STYLE
Terminology: Executive, concise.
Structure: One summary paragraph.
Perspective: Third-person.
"""

# ------------------------------------------------------------------------------

patient_baseline_system = f"""SYSTEM ROLE
Compassionate Medical Liaison explaining results to a patient.

OBJECTIVE
Explain the result using a "Scale/Balance" analogy (derived from Feature Importance).

INPUT DATA
You will receive Prediction, XAI Weights, and Patient Data.

{BENIGN_INSTRUCTION}

CONSTRAINTS
- NO JARGON (No "SHAP", "Coefficients").
- NO "Checklists" or "Steps" (Use "Weights" or "Influence").

NARRATIVE STRUCTURE
1. Result Context: Clear and gentle.
2. The "Why" (The Scale Analogy): "Imagine a scale. The system noticed that the Size and Texture stood out the most. These factors combined to tip the calculation towards this result."
3. Next Steps: Encourage discussion.

TONE & STYLE
Terminology: Simple visual descriptions.
Structure: Warm, reassuring letter format.
Perspective: First-person ("We", "The system").
"""

## 4. Experimental Execution & Comparative Analysis
In this final section, we put our architecture to the test. To maintain a clean and readable experimental environment, the core logic has been abstracted into our custom `nle-framework-pappppx` package, specifically following the Separation of Concerns principle.

By importing the `NLEModel` and `NLEAdapter` classes here, we let the package handle the heavy lifting under the hood:
* **LLM Memory Management (`NLEModel`):** Utilizing a Singleton pattern to cache the Llama-3 model in the GPU, preventing Out-of-Memory errors and avoiding redundant loading times.
* **Automated XAI Processing (`NLEAdapter`):** Acting as dedicated configuration profiles to extract and compute metrics from SHAP, LIME, and Araucana simultaneously.
* **Prompt Orchestration (`NLEAdapter`):** Dynamically constructing the final prompt by fusing XAI contexts with templates, while strictly hiding the ground truth from the LLM to prevent data leakage.

**Evaluation Metric (Qualitative):**
We isolate a set of patient instances (e.g., ID 2, 3, and 8, representing diverse tumor profiles) and generate distinct Natural Language Explanations for each. We aim to observe the shift in the LLM's reasoning structure:
* **Experiment A (Baseline):** Does the model rely heavily on "tipping the scale" (feature weights from SHAP/LIME) without clear logical boundaries?
* **Experiment B (Hybrid):** Does the model successfully anchor its explanation on a strict threshold (Araucana symbolic logic rules) while maintaining clinical fluency?

In [ ]:
# BLOCK 4: Batch Comparative Experiment (Patients 2, 3, 8)
# Generates 6 reports per patient (3 Hybrid vs. 3 Baseline)
# Optimizes visualization by showing plots only once per patient.

from nle_core.NLEFramework import NLEModel, NLEAdapter

# 1. Initialize Framework Model (Loads LLM into GPU once)
nlem = NLEModel(llm_model_name="unsloth/llama-3-8b-Instruct")

# 2. Register All Adapters (Hybrid & Baseline)
# We utilize the UNIVERSAL_USER_TEMPLATE defined in Block 4

# --- Hybrid Set (Logic-Based) ---
detailed_hybrid_adapter = NLEAdapter(
    system_prompt=detailed_hybrid_system,
    user_template=UNIVERSAL_USER_TEMPLATE,
    nle_model=nlem,
    class_names={0: "Benign", 1: "Malignant"}
)

executive_hybrid_adapter = NLEAdapter(
    system_prompt=executive_hybrid_system,
    user_template=UNIVERSAL_USER_TEMPLATE,
    nle_model=nlem,
    class_names={0: "Benign", 1: "Malignant"}
)

patient_hybrid_adapter = NLEAdapter(
    system_prompt=patient_hybrid_system,
    user_template=UNIVERSAL_USER_TEMPLATE,
    nle_model=nlem,
    class_names={0: "Benign", 1: "Malignant"}
)

# --- Baseline Set (Weight-Based) ---
detailed_baseline_adapter = NLEAdapter(
    system_prompt=detailed_baseline_system,
    user_template=UNIVERSAL_USER_TEMPLATE,
    nle_model=nlem,
    class_names={0: "Benign", 1: "Malignant"}
)

executive_baseline_adapter = NLEAdapter(
    system_prompt=executive_baseline_system,
    user_template=UNIVERSAL_USER_TEMPLATE,
    nle_model=nlem,
    class_names={0: "Benign", 1: "Malignant"}
)

patient_baseline_adapter = NLEAdapter(
    system_prompt=patient_baseline_system,
    user_template=UNIVERSAL_USER_TEMPLATE,
    nle_model=nlem,
    class_names={0: "Benign", 1: "Malignant"}
)


# 3. Define Patients of Interest
# ID 2, 3, 8 are selected to show diverse tumor profiles
target_indices = [2, 3, 8]

# 4. Batch Execution Loop
for idx in target_indices:
    instance = X_test.iloc[[idx]]
    true_label = y_test.iloc[[idx]]

    print("\n" + "#" * 100)
    print(f"###   ANALYSIS FOR PATIENT ID: {idx}   ###")
    print("#" * 100)

    # ==========================================================================
    # PHASE 1: HYBRID GENERATION (Includes Plots)
    # We run this FIRST with plot=True to show SHAP + LIME + ARAUCANA once.
    # ==========================================================================
    print("\n" + ">>> GENERATING VISUAL EVIDENCE (SHAP, LIME, ARAUCANA)...")

    # 1.1 Detailed Technical Report (Hybrid) -> PLOTS ON
    print("\n" + "="*40 + " HYBRID REPORT: DETAILED " + "="*40)
    print(detailed_hybrid_adapter.generate_explanation(
        black_box_model=model, 
        X_train=X_train, 
        instance=instance, 
        y_true=true_label,
        explainer_type="shap+lime+araucana",
        plot=True  # <--- PLOTS SHOWN HERE (Once per patient)
    ))

    # 1.2 Executive Summary (Hybrid) -> Plots Off
    print("\n" + "="*40 + " HYBRID REPORT: EXECUTIVE " + "="*40)
    print(executive_hybrid_adapter.generate_explanation(
        black_box_model=model, 
        X_train=X_train, 
        instance=instance, 
        y_true=true_label,
        explainer_type="shap+lime+araucana",
        plot=False
    ))

    # 1.3 Patient Letter (Hybrid) -> Plots Off
    print("\n" + "="*40 + " HYBRID REPORT: PATIENT " + "="*40)
    print(patient_hybrid_adapter.generate_explanation(
        black_box_model=model, 
        X_train=X_train, 
        instance=instance, 
        y_true=true_label,
        explainer_type="shap+lime+araucana",
        plot=False
    ))

    # ==========================================================================
    # PHASE 2: BASELINE GENERATION (Text Only)
    # We use plot=False because we already saw SHAP/LIME in Phase 1.
    # ==========================================================================

    # 2.1 Detailed Technical Report (Baseline)
    print("\n" + "="*40 + " BASELINE REPORT: DETAILED " + "="*40)
    print(detailed_baseline_adapter.generate_explanation(
        black_box_model=model, 
        X_train=X_train, 
        instance=instance, 
        y_true=true_label,
        explainer_type="shap+lime",        # Only Weights
        plot=False                         # Avoid duplicate plots
    ))

    # 2.2 Executive Summary (Baseline)
    print("\n" + "="*40 + " BASELINE REPORT: EXECUTIVE " + "="*40)
    print(executive_baseline_adapter.generate_explanation(
        black_box_model=model, 
        X_train=X_train, 
        instance=instance, 
        y_true=true_label,
        explainer_type="shap+lime",
        plot=False
    ))

    # 2.3 Patient Letter (Baseline)
    print("\n" + "="*40 + " BASELINE REPORT: PATIENT " + "="*40)
    print(patient_baseline_adapter.generate_explanation(
        black_box_model=model, 
        X_train=X_train, 
        instance=instance, 
        y_true=true_label,
        explainer_type="shap+lime",
        plot=False
    ))

    print("\n" + "_"*100 + "\n")